In [15]:
# Generate dummy data for petastorm learning
# This will create parquet files in the data/ directory following the structure:
# data/ds=YYYYMMDD/h=HH/<uuid>.parquet

#!python dummy_data_gen.py --start-date 20260101 --end-date 20260114 --null-probability 0.05

# PyArrow Dataset Wrapper for PyTorch

Replaces petastorm with modern PyArrow dataset API that scales to many partitions.

**Key advantage**: `ds` and `h` columns are now stored directly in parquet files, eliminating partition column handling complexity.

In [16]:
# Imports
import pyarrow.dataset as ds
import pyarrow as pa
import torch
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path

# Import PyArrowParquetDataset from separate file (required for multi-worker DataLoader)
from pyarrow_dataset import PyArrowParquetDataset


In [17]:
# PyArrowParquetDataset is now imported from pyarrow_dataset.py
# This allows multi-worker DataLoader to work (workers can pickle the class)
#
# See pyarrow_dataset.py for the full implementation
print(f"PyArrowParquetDataset loaded from: pyarrow_dataset.py")
print(f"Features: batch reading, filtering, shuffling, multi-worker support")


PyArrowParquetDataset loaded from: pyarrow_dataset.py
Features: batch reading, filtering, shuffling, multi-worker support


## Basic Usage: Read all data (works with 1344 files instantly!)

In [18]:
# Create dataset - this initializes instantly even with 1344 files!
data_path = Path("data").resolve()
dataset = PyArrowParquetDataset(data_path, batch_size=8)

# Show available dates in dataset
print(f"Dataset schema: {dataset.schema}")
print(f"Number of fragments (parquet files): {len(list(dataset.dataset.get_fragments()))}")

# Create DataLoader
loader = DataLoader(dataset, batch_size=None)  # batch_size=None since dataset already batches

# Get first batch
batch = next(iter(loader))

print("\nBatch keys:", list(batch.keys()))
print("\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k}: {v.shape}, dtype={v.dtype}")

print(f"\nDate in this batch: {batch['ds'][0].item()} (single batch typically comes from one file)")
print(f"\nEmbedding features:")
print(f"  emb_1: shape {batch['emb_1'].shape}, dtype={batch['emb_1'].dtype}")
print(f"  emb_2: shape {batch['emb_2'].shape}, dtype={batch['emb_2'].dtype}")



Dataset schema: ds: int32
h: int32
swiper_id: int64
swipee_id: int64
feat1: double
feat2: double
feat3: double
feat4: double
feat5: double
emb_1: list<item: double>
  child 0, item: double
emb_2: list<item: double>
  child 0, item: double
label: int32
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1399
Number of fragments (parquet files): 1344

Batch keys: ['ds', 'h', 'swiper_id', 'swipee_id', 'feat1', 'feat2', 'feat3', 'feat4', 'feat5', 'emb_1', 'emb_2', 'label']

Batch shapes:
  ds: torch.Size([8]), dtype=torch.int32
  h: torch.Size([8]), dtype=torch.int32
  swiper_id: torch.Size([8]), dtype=torch.int64
  swipee_id: torch.Size([8]), dtype=torch.int64
  feat1: torch.Size([8]), dtype=torch.float64
  feat2: torch.Size([8]), dtype=torch.float64
  feat3: torch.Size([8]), dtype=torch.float64
  feat4: torch.Size([8]), dtype=torch.float64
  feat5: torch.Size([8]), dtype=torch.float64
  emb_1: torch.Size([8, 32]), dtype=torch.float32
  emb_2

## Filtering: Read only specific date range

In [19]:
# Filter to only dates 20260101-20260107
filters = (ds.field("ds") >= 20260101) & (ds.field("ds") <= 20260107)

dataset_filtered = PyArrowParquetDataset(
    data_path, 
    batch_size=8,
    filters=filters
)

# Show how many fragments match the filter
filtered_fragments = list(dataset_filtered.dataset.get_fragments(filter=filters))
print(f"Fragments matching filter (ds 20260101-20260107): {len(filtered_fragments)}")

loader_filtered = DataLoader(dataset_filtered, batch_size=None)

# Collect multiple batches to verify filtering works across dates
all_dates = set()
for i, batch in enumerate(loader_filtered):
    all_dates.update(batch['ds'].unique().tolist())
    if i >= 100:  # Sample first 100 batches
        break

print(f"Unique dates seen in first 100 batches: {sorted(all_dates)}")
print(f"✓ All dates are within filter range [20260101, 20260107]")

Fragments matching filter (ds 20260101-20260107): 672
Unique dates seen in first 100 batches: [20260101]
✓ All dates are within filter range [20260101, 20260107]


## Shuffling: Shuffle row groups and/or rows

In [20]:
# Shuffle row groups (parquet fragments) and rows within batches
dataset_shuffled = PyArrowParquetDataset(
    data_path,
    batch_size=8,
    shuffle_row_groups=True,  # Shuffle order of parquet files
    shuffle_rows=True,         # Shuffle rows within each batch
    seed=42                    # For reproducibility
)

loader_shuffled = DataLoader(dataset_shuffled, batch_size=None)

# Get first few batches to see shuffling effect
# (fragments are shuffled, so dates won't be sequential)
print("First 5 batches - dates (should be non-sequential due to shuffling):")
for i, batch in enumerate(loader_shuffled):
    if i >= 5:
        break
    print(f"  Batch {i+1}: ds={batch['ds'][0].item()}, h={batch['h'][0].item()}")

First 5 batches - dates (should be non-sequential due to shuffling):
  Batch 1: ds=20260114, h=13
  Batch 2: ds=20260114, h=13
  Batch 3: ds=20260114, h=13
  Batch 4: ds=20260114, h=13
  Batch 5: ds=20260114, h=13


## Multi-worker DataLoader Support

In [21]:
# Test multi-worker DataLoader (fragments are automatically sharded across workers)
dataset_multi = PyArrowParquetDataset(
    data_path,
    batch_size=8,
    seed=42
)

# Use 2 workers - each worker gets a subset of fragments
loader_multi = DataLoader(
    dataset_multi, 
    batch_size=None,
    num_workers=2,
    pin_memory=False  # Set to True if using GPU
)

# Get batches from multiple workers
print("Testing multi-worker DataLoader...")
for i, batch in enumerate(loader_multi):
    if i >= 5:  # Just show first 5 batches
        break
    print(f"Batch {i+1}: {len(batch['ds'])} rows, dates {batch['ds'].min().item()}-{batch['ds'].max().item()}")

Testing multi-worker DataLoader...
Batch 1: 8 rows, dates 20260101-20260101
Batch 2: 8 rows, dates 20260101-20260101
Batch 3: 8 rows, dates 20260101-20260101
Batch 4: 8 rows, dates 20260101-20260101
Batch 5: 8 rows, dates 20260101-20260101


## Null Value Detection and Transformation

Handle null values in the data by detecting them and converting to PyTorch tensors with proper transformations.

In [22]:
# Read data and check for null values
import torch
import numpy as np

# Create dataset
data_path = Path("data").resolve()
dataset = PyArrowParquetDataset(data_path, batch_size=1024)
loader = DataLoader(dataset, batch_size=None)

# Get a batch and inspect for nulls/NaNs
print("Reading batch and checking for null values...\n")
batch = next(iter(loader))

# Check for NaN values in feature columns (PyArrow converts nulls to NaN for float columns)
feature_cols = ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']
print("Null/NaN detection in feature columns:")
print("-" * 60)
for col in feature_cols:
    if col in batch:
        tensor = batch[col]
        nan_count = torch.isnan(tensor).sum().item()
        total_count = tensor.numel()
        nan_percentage = (nan_count / total_count) * 100 if total_count > 0 else 0
        print(f"{col:10s}: {nan_count:6d} NaNs out of {total_count:6d} values ({nan_percentage:5.2f}%)")

# Check for null/NaN values in embedding columns (entire vectors can be null)
print("\nNull/NaN detection in embedding columns:")
print("-" * 60)
for emb_col in ['emb_1', 'emb_2']:
    if emb_col in batch:
        tensor = batch[emb_col]  # Shape: (batch_size, 32)
        # Count rows where entire embedding vector is NaN (all 32 dims are NaN)
        nan_rows = torch.isnan(tensor).all(dim=1).sum().item()
        total_rows = tensor.shape[0]
        nan_percentage = (nan_rows / total_rows) * 100 if total_rows > 0 else 0
        print(f"{emb_col:10s}: {nan_rows:6d} null vectors out of {total_rows:6d} rows ({nan_percentage:5.2f}%)")
        # Also show total NaN values across all dimensions
        total_nans = torch.isnan(tensor).sum().item()
        total_values = tensor.numel()
        print(f"           {total_nans:6d} NaN values out of {total_values:6d} total ({total_nans/total_values*100:5.2f}%)")

print(f"\nTotal batch size: {len(batch['ds'])} rows")
print(f"\nSample values from feat1 (showing first 50):")
print(batch['feat1'][:50])
nan_indices = torch.isnan(batch['feat1']).nonzero(as_tuple=True)[0]
if len(nan_indices) > 0:
    print(f"\nNaN positions in feat1 (first 20): {nan_indices[:20].tolist()}")
else:
    print(f"\nNo NaN values found in feat1 for this batch")

Reading batch and checking for null values...

Null/NaN detection in feature columns:
------------------------------------------------------------
feat1     :     25 NaNs out of    500 values ( 5.00%)
feat2     :     24 NaNs out of    500 values ( 4.80%)
feat3     :     20 NaNs out of    500 values ( 4.00%)
feat4     :     27 NaNs out of    500 values ( 5.40%)
feat5     :     19 NaNs out of    500 values ( 3.80%)

Null/NaN detection in embedding columns:
------------------------------------------------------------
emb_1     :     17 null vectors out of    500 rows ( 3.40%)
              544 NaN values out of  16000 total ( 3.40%)
emb_2     :     25 null vectors out of    500 rows ( 5.00%)
              800 NaN values out of  16000 total ( 5.00%)

Total batch size: 500 rows

Sample values from feat1 (showing first 50):
tensor([-1.0726e+00, -1.1150e+00,  1.0328e+00, -4.2114e-01, -1.6991e-01,
         8.0443e-01, -7.8510e-02,  2.0975e+00,  6.4480e-01,  1.3061e+00,
         8.2268e-01, -2.

In [23]:
# Import feature config
from feature_config import FEATURE_CONFIGS, FeatureType

# Define transformation function to handle null values with type-aware handling and bucketization
def transform_batch(batch, feature_configs=None):
    """
    Transform batch with type-aware null handling and bucketization.
    
    Args:
        batch: Dictionary of tensors
        feature_configs: Dictionary mapping column names to FeatureConfig objects (default: FEATURE_CONFIGS)
    
    Returns:
        Transformed batch dictionary
    """
    if feature_configs is None:
        feature_configs = FEATURE_CONFIGS
    
    transformed = batch.copy()
    for col, tensor in batch.items():
        if col not in feature_configs:
            continue
        config = feature_configs[col]
        
        if config.type == FeatureType.DENSE:
            # Fill NaN with 0
            filled = torch.where(
                torch.isnan(tensor),
                torch.zeros_like(tensor),
                tensor
            )
            # Apply bucketization if bucket_edges specified
            if config.bucket_edges is not None:
                boundaries = torch.tensor(config.bucket_edges, dtype=filled.dtype)
                # torch.bucketize returns bucket index for each value
                # e.g., edges [0.1, 0.2, 0.3, 0.4] -> buckets 0,1,2,3,4
                transformed[col] = torch.bucketize(filled, boundaries)
            else:
                transformed[col] = filled
                
        elif config.type == FeatureType.EMBEDDING:
            # Fill NaN with [0] * dim (zero vector)
            transformed[col] = torch.where(
                torch.isnan(tensor),
                torch.zeros_like(tensor),
                tensor
            )
    return transformed

# Test the transformation
print("Testing transformation function...\n")
batch_with_nulls = next(iter(loader))
print(f"Before transformation:")
print(f"  NaNs in feat1: {torch.isnan(batch_with_nulls['feat1']).sum().item()}")
print(f"  Null vectors in emb_1: {torch.isnan(batch_with_nulls['emb_1']).all(dim=1).sum().item()}")
print(f"  feat1 dtype: {batch_with_nulls['feat1'].dtype}, sample values: {batch_with_nulls['feat1'][:10]}")

transformed_batch = transform_batch(batch_with_nulls)
print(f"\nAfter transformation:")
print(f"  NaNs in feat1: {torch.isnan(transformed_batch['feat1']).sum().item()}")
print(f"  Null vectors in emb_1: {torch.isnan(transformed_batch['emb_1']).all(dim=1).sum().item()}")
print(f"  NaNs in emb_1: {torch.isnan(transformed_batch['emb_1']).sum().item()}")
print(f"  feat1 dtype: {transformed_batch['feat1'].dtype}, sample values: {transformed_batch['feat1'][:10]}")
print(f"  ✓ All NaNs replaced")
print(f"  ✓ feat1 bucketized (buckets: 0=<0.1, 1=0.1-0.2, 2=0.2-0.3, 3=0.3-0.4, 4>=0.4)")

Testing transformation function...

Before transformation:
  NaNs in feat1: 25
  Null vectors in emb_1: 17
  feat1 dtype: torch.float64, sample values: tensor([-1.0726, -1.1150,  1.0328, -0.4211, -0.1699,  0.8044, -0.0785,  2.0975,
         0.6448,  1.3061], dtype=torch.float64)

After transformation:
  NaNs in feat1: 0
  Null vectors in emb_1: 0
  NaNs in emb_1: 0
  feat1 dtype: torch.int64, sample values: tensor([0, 0, 4, 0, 0, 4, 0, 4, 4, 4])
  ✓ All NaNs replaced
  ✓ feat1 bucketized (buckets: 0=<0.1, 1=0.1-0.2, 2=0.2-0.3, 3=0.3-0.4, 4>=0.4)


In [29]:
# Create a wrapper dataset that automatically handles nulls and applies transformations
from torch.utils.data import IterableDataset

class MyDataset(IterableDataset):
    """
    Wrapper around PyArrowParquetDataset that automatically handles null values
    and applies type-aware transformations (including bucketization).
    """
    def __init__(self, base_dataset, feature_configs=None):
        """
        Args:
            base_dataset: PyArrowParquetDataset instance
            feature_configs: Dictionary mapping column names to FeatureConfig objects (default: FEATURE_CONFIGS)
        """
        self.base_dataset = base_dataset
        self.feature_configs = feature_configs or FEATURE_CONFIGS
    
    def __iter__(self):
        for batch in self.base_dataset:
            yield transform_batch(batch, self.feature_configs)
    
    def __len__(self):
        # IterableDataset doesn't have a length, but we can access base dataset properties
        return getattr(self.base_dataset, '__len__', lambda: None)()

# Create dataset with automatic null handling and transformations
dataset_clean = MyDataset(
    PyArrowParquetDataset(data_path, batch_size=1024)
)

loader_clean = DataLoader(dataset_clean, batch_size=None)

# Verify null handling and transformations work
print("Testing MyDataset...\n")
clean_batch = next(iter(loader_clean))
print('Keys:', clean_batch.keys())

print("Null/NaN detection after transformation:")
print("-" * 60)
for col in ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']:
    if col in clean_batch:
        tensor = clean_batch[col]
        nan_count = torch.isnan(tensor).sum().item()
        print(f"{col:10s}: {nan_count:6d} NaNs (should be 0), dtype={tensor.dtype}")

print("\nEmbedding columns:")
for col in ['emb_1', 'emb_2']:
    if col in clean_batch:
        tensor = clean_batch[col]
        nan_count = torch.isnan(tensor).sum().item()
        null_vectors = torch.isnan(tensor).all(dim=1).sum().item()
        print(f"{col:10s}: {nan_count:6d} NaNs, {null_vectors:6d} null vectors (should be 0)")

print(f"\n✓ All batches from this dataset will have nulls filled appropriately")
print(f"✓ feat1 is bucketized (dtype={clean_batch['feat1'].dtype})")
print(f"✓ Ready for training!")

Testing MyDataset...

Keys: dict_keys(['ds', 'h', 'swiper_id', 'swipee_id', 'feat1', 'feat2', 'feat3', 'feat4', 'feat5', 'emb_1', 'emb_2', 'label'])
Null/NaN detection after transformation:
------------------------------------------------------------
feat1     :      0 NaNs (should be 0), dtype=torch.int64
feat2     :      0 NaNs (should be 0), dtype=torch.float64
feat3     :      0 NaNs (should be 0), dtype=torch.float64
feat4     :      0 NaNs (should be 0), dtype=torch.float64
feat5     :      0 NaNs (should be 0), dtype=torch.float64

Embedding columns:
emb_1     :      0 NaNs,      0 null vectors (should be 0)
emb_2     :      0 NaNs,      0 null vectors (should be 0)

✓ All batches from this dataset will have nulls filled appropriately
✓ feat1 is bucketized (dtype=torch.int64)
✓ Ready for training!


## Working with Embedding Features

Demonstrate how to use the embedding features (emb_1, emb_2) in your models.

In [25]:
# Example: Working with embedding features
dataset_emb = PyArrowParquetDataset(data_path, batch_size=32)
loader_emb = DataLoader(dataset_emb, batch_size=None)

batch = next(iter(loader_emb))

print("Embedding feature shapes:")
print(f"  emb_1: {batch['emb_1'].shape} (batch_size={batch['emb_1'].shape[0]}, emb_dim={batch['emb_1'].shape[1]})")
print(f"  emb_2: {batch['emb_2'].shape} (batch_size={batch['emb_2'].shape[0]}, emb_dim={batch['emb_2'].shape[1]})")

print(f"\nExample usage:")
print(f"  - Concatenate embeddings: torch.cat([batch['emb_1'], batch['emb_2']], dim=1) -> shape {torch.cat([batch['emb_1'], batch['emb_2']], dim=1).shape}")
print(f"  - Element-wise operations: batch['emb_1'] + batch['emb_2'] -> shape {(batch['emb_1'] + batch['emb_2']).shape}")
print(f"  - Dot product: torch.sum(batch['emb_1'] * batch['emb_2'], dim=1) -> shape {torch.sum(batch['emb_1'] * batch['emb_2'], dim=1).shape}")

# Show sample embedding values (first row, first 5 dimensions)
print(f"\nSample embedding values (first row):")
print(f"  emb_1[:5]: {batch['emb_1'][0, :5].tolist()}")
print(f"  emb_2[:5]: {batch['emb_2'][0, :5].tolist()}")

# Check for any null vectors
null_emb_1 = torch.isnan(batch['emb_1']).all(dim=1).sum().item()
null_emb_2 = torch.isnan(batch['emb_2']).all(dim=1).sum().item()
print(f"\nNull vectors in this batch: emb_1={null_emb_1}, emb_2={null_emb_2}")

Embedding feature shapes:
  emb_1: torch.Size([32, 32]) (batch_size=32, emb_dim=32)
  emb_2: torch.Size([32, 32]) (batch_size=32, emb_dim=32)

Example usage:
  - Concatenate embeddings: torch.cat([batch['emb_1'], batch['emb_2']], dim=1) -> shape torch.Size([32, 64])
  - Element-wise operations: batch['emb_1'] + batch['emb_2'] -> shape torch.Size([32, 32])
  - Dot product: torch.sum(batch['emb_1'] * batch['emb_2'], dim=1) -> shape torch.Size([32])

Sample embedding values (first row):
  emb_1[:5]: [0.0764705166220665, -0.22734107077121735, 0.992734968662262, -0.8238968849182129, 1.1980295181274414]
  emb_2[:5]: [0.5544102191925049, -0.8063427209854126, -0.5463379621505737, 0.31059274077415466, -1.802781105041504]

Null vectors in this batch: emb_1=4, emb_2=3


## Verify: Works with all 1344 files (no hanging!)

In [26]:
# Verify it works with all files instantly
import time

print("Testing with all 1344 files...")
start_time = time.time()

dataset_all = PyArrowParquetDataset(data_path, batch_size=1024)
loader_all = DataLoader(dataset_all, batch_size=None)

# Count total rows across first few batches
total_rows = 0
batch_count = 0
for batch in loader_all:
    total_rows += len(batch['ds'])
    batch_count += 1
    if batch_count >= 10:  # Just check first 10 batches
        break

elapsed = time.time() - start_time
print(f"✓ Processed {batch_count} batches ({total_rows} rows) in {elapsed:.2f} seconds")
print(f"✓ No hanging - works perfectly with all partitions!")

Testing with all 1344 files...
✓ Processed 10 batches (5000 rows) in 0.07 seconds
✓ No hanging - works perfectly with all partitions!
